# Behind a Transformers pipeline

In this notebook, we manually reproduce a sentiment-analysis pipeline:

1. Tokenize raw text.
2. Convert tokens to IDs and tensors.
3. Run the base Transformer to inspect hidden states.
4. Run a sequence-classification head to obtain logits.
5. Apply softmax and map class IDs to labels.
6. Compare the manual result with `pipeline()`.

In [ ]:
!pip install -q transformers

In [ ]:
import torch
from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    pipeline,
)

checkpoint = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
texts = [
    "I've been waiting for a Hugging Face course my whole life.",
    "I dislike this lesson very much!",
]

print("PyTorch version:", torch.__version__)
print("Number of texts:", len(texts))

## 1. Tokenization

A model cannot process raw strings directly. The tokenizer splits text into tokens, maps each token to an integer ID, adds special tokens, pads shorter sequences, and creates an attention mask.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

for text in texts:
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    print("TEXT:", text)
    print("TOKENS:", tokens)
    print("TOKEN IDS:", token_ids)
    print()

In [ ]:
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
)

print(inputs)
print("input_ids shape:", inputs["input_ids"].shape)
print("attention_mask shape:", inputs["attention_mask"].shape)

In [ ]:
for row in range(len(texts)):
    ids = inputs["input_ids"][row]
    mask = inputs["attention_mask"][row]
    tokens = tokenizer.convert_ids_to_tokens(ids)

    print(f"SEQUENCE {row + 1}")
    for token, token_id, attention in zip(tokens, ids.tolist(), mask.tolist()):
        print(f"{token:15} id={token_id:5} attention={attention}")
    print()

An attention-mask value of `1` marks a real token. A value of `0` marks padding that should not influence the model's contextual representation.

## 2. Base Transformer and hidden states

`AutoModel` loads the base Transformer without a task-specific classification head. It produces a contextual vector for every token.

In [ ]:
base_model = AutoModel.from_pretrained(checkpoint)
base_model.eval()

with torch.no_grad():
    base_outputs = base_model(**inputs)

hidden_states = base_outputs.last_hidden_state
print("Hidden-state shape:", hidden_states.shape)
print("Batch size:", hidden_states.shape[0])
print("Sequence length:", hidden_states.shape[1])
print("Hidden size:", hidden_states.shape[2])
print("First token, first 10 values:", hidden_states[0, 0, :10])

The hidden-state tensor has three dimensions: batch size, sequence length, and hidden size. These contextual vectors are useful features, but they are not yet sentiment scores.

## 3. Classification head and logits

`AutoModelForSequenceClassification` loads the same base architecture plus a head trained to produce one score for each sentiment label.

In [ ]:
classifier_model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
classifier_model.eval()

with torch.no_grad():
    outputs = classifier_model(**inputs)

logits = outputs.logits
print("Logits:")
print(logits)
print("Logits shape:", logits.shape)
print("Label mapping:", classifier_model.config.id2label)

Logits are raw scores. They can be positive or negative and do not add up to 1.

## 4. Postprocessing

Softmax converts each row of logits into probabilities. The index of the largest probability is mapped to a readable label using the model configuration.

In [ ]:
probabilities = torch.nn.functional.softmax(logits, dim=-1)
predicted_class_ids = probabilities.argmax(dim=-1)

manual_results = []
for text, class_id, scores in zip(texts, predicted_class_ids, probabilities):
    class_id = class_id.item()
    result = {
        "text": text,
        "label": classifier_model.config.id2label[class_id],
        "score": scores[class_id].item(),
    }
    manual_results.append(result)

print("Probabilities:")
print(probabilities)
print("Row sums:", probabilities.sum(dim=-1))
print("Manual results:")
for result in manual_results:
    print(result)

## 5. Compare with pipeline()

In [ ]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=checkpoint,
    tokenizer=checkpoint,
)

pipeline_results = sentiment_pipeline(texts)

for manual, automatic in zip(manual_results, pipeline_results):
    print("Manual:  ", manual["label"], round(manual["score"], 6))
    print("Pipeline:", automatic["label"], round(automatic["score"], 6))
    print()

assert [item["label"] for item in manual_results] == [
    item["label"] for item in pipeline_results
]
print("The manual and pipeline labels match.")

## 6. Student challenge

Replace the examples below with two or more sentences of your own. Repeat the manual steps and check that the manual labels match `pipeline()`.

In [ ]:
my_texts = [
    "The live demonstration made the concept easy to understand.",
    "The instructions were incomplete and difficult to follow.",
]

my_inputs = tokenizer(
    my_texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
)

with torch.no_grad():
    my_logits = classifier_model(**my_inputs).logits

my_probabilities = torch.nn.functional.softmax(my_logits, dim=-1)
my_class_ids = my_probabilities.argmax(dim=-1)

for text, class_id, scores in zip(my_texts, my_class_ids, my_probabilities):
    class_id = class_id.item()
    print(text)
    print(classifier_model.config.id2label[class_id], scores[class_id].item())
    print()

print("Pipeline check:", sentiment_pipeline(my_texts))

## Key takeaway

`pipeline()` is convenient, but it is not magic. For sentiment analysis it performs the same major stages you reproduced:

`raw text -> tokenizer -> input IDs and attention mask -> Transformer -> classification head -> logits -> softmax -> labels`